In [1]:
# Cell 1 — confirm environment
import torch
import transformers, datasets, peft
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)

torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti
transformers: 5.16.1
datasets: 5.0.1
peft: 0.20.0


In [2]:
# Cell 2 — load LEDGAR and inspect
from datasets import load_dataset

# LEDGAR: contract clause classification, 100 clause types.
# Part of the LexGLUE benchmark.
dataset = load_dataset("coastalcph/lex_glue", "ledgar")

print("Splits:", {k: len(v) for k, v in dataset.items()})
print("\nColumns:", dataset["train"].column_names)

# Look at one real example
example = dataset["train"][0]
print("\n--- Example clause ---")
print("Text:", example["text"][:400], "...")
print("Label (numeric):", example["label"])

# The 100 label names
label_names = dataset["train"].features["label"].names
print("\nNumber of labels:", len(label_names))
print("First 15 label names:", label_names[:15])

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

d:\Projects\Legal Clause Classifier\legal-clause-classifier\venv\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AryanPC\.cache\huggingface\hub\datasets--coastalcph--lex_glue. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ledgar/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.9MB            

ledgar/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ledgar/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.31MB            

...

ledgar/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Splits: {'train': 60000, 'test': 10000, 'validation': 10000}

Columns: ['text', 'label']

--- Example clause ---
Text: Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection. ...
Label (numeric): 97

Number of labels: 100
First 15 label names: ['Adjustments', 'Agreements', 'Amendments', 'Anti-Corruption Laws', 'Applicable Laws', 'Approvals', 'Arbitration', 'Assignments', 'Assigns', 'Authority', 'Authorizations', 'Base Salary', 'Benefits', 'Binding Effects', 'Books']


In [3]:
# Cell 3 — label distribution (decides our metric)
from collections import Counter

label_names = dataset["train"].features["label"].names
counts = Counter(dataset["train"]["label"])

# Most and least common clause types
sorted_counts = counts.most_common()
print("Most common clause types:")
for label_id, n in sorted_counts[:5]:
    print(f"  {label_names[label_id]:25} {n:>6}  ({100*n/60000:.1f}%)")

print("\nLeast common clause types:")
for label_id, n in sorted_counts[-5:]:
    print(f"  {label_names[label_id]:25} {n:>6}  ({100*n/60000:.1f}%)")

# Imbalance summary
most = sorted_counts[0][1]
least = sorted_counts[-1][1]
print(f"\nMost common: {most} | Least common: {least} | Imbalance ratio: {most/least:.1f}x")
print(f"Avg per class: {60000/100:.0f}")

Most common clause types:
  Governing Laws              3167  (5.3%)
  Notices                     2493  (4.2%)
  Counterparts                2427  (4.0%)
  Entire Agreements           2340  (3.9%)
  Severability                1808  (3.0%)

Least common clause types:
  Sanctions                    118  (0.2%)
  Anti-Corruption Laws         106  (0.2%)
  Qualifications                47  (0.1%)
  Assigns                       31  (0.1%)
  Books                         23  (0.0%)

Most common: 3167 | Least common: 23 | Imbalance ratio: 137.7x
Avg per class: 600


In [7]:
# Cell 4 — tokenize the dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=384)

# Tokenize all splits
tokenized = dataset.map(tokenize, batched=True)
print("Tokenized. Columns now:", tokenized["train"].column_names)

# Check token-length distribution to confirm 384 is enough
import numpy as np
lengths = [len(tokenizer(t, truncation=False)["input_ids"]) for t in dataset["train"]["text"][:2000]]
print(f"Token lengths (sample of 2000): mean {np.mean(lengths):.0f}, "
      f"95th pct {np.percentile(lengths,95):.0f}, max {np.max(lengths)}")
print(f"Clauses over 384 tokens: {100*np.mean(np.array(lengths)>384):.1f}%")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (693 > 512). Running this sequence through the model will result in indexing errors


Tokenized. Columns now: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
Token lengths (sample of 2000): mean 145, 95th pct 385, max 1169
Clauses over 384 tokens: 5.1%


In [8]:
# Cell 5 — load DistilBERT for 100-class classification, wrap with LoRA
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

label_names = dataset["train"].features["label"].names
num_labels = len(label_names)  # 100

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 962,404 || all params: 67,992,776 || trainable%: 1.4155
